<a href="https://colab.research.google.com/github/rdelhibabu/Optimization_of_QCNNN/blob/main/Optimization_QCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this in the first Colab cell
!pip install pennylane torch torchvision scikit-learn matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 780.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 61.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import copy

# Set random seeds for reproducible research
torch.manual_seed(42)
np.random.seed(42)

# HA-QFed System Constraints (from Table V/VI)
N_QUBITS = 8
N_FEATURES = 256 # 2^8
NUM_CLIENTS = 10
DIRICHLET_ALPHA = 0.1
LOCAL_EPOCHS = 3
LEARNING_RATE = 0.02
PROXIMAL_MU = 0.15
GLOBAL_ROUNDS = 50

In [4]:
def prepare_non_iid_cifar10(num_clients, alpha, n_components=256):
    print("Downloading and preprocessing CIFAR-10...")
    # Using a binary subset (e.g., classes 0 and 1) for rapid QCNN benchmarking
    transform = transforms.Compose([
        transforms.Grayscale(),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    trainset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

    # Filter for binary classification (Classes 0 and 1) to match single-qubit measurement
    idx = (np.array(trainset.targets) == 0) | (np.array(trainset.targets) == 1)
    trainset.data = trainset.data[idx]
    trainset.targets = np.array(trainset.targets)[idx]

    # Flatten and PCA to bridge classical-quantum dimensional gap
    print(f"Applying PCA to reduce classical features to {n_components} dimensions...")
    x_train = trainset.data.reshape(len(trainset.data), -1) / 255.0
    pca = PCA(n_components=n_components)
    x_pca = pca.fit_transform(x_train)
    y_train = trainset.targets

    # Dirichlet Non-IID Partitioning
    client_data = {i: {'x': [], 'y': []} for i in range(num_clients)}
    classes = np.unique(y_train)

    for c in classes:
        idx_c = np.where(y_train == c)[0]
        np.random.shuffle(idx_c)
        # Allocate proportions based on Dirichlet distribution
        proportions = np.random.dirichlet(np.repeat(alpha, num_clients))
        proportions = np.array([p * (len(idx_c) < len(y_train)) for p in proportions])
        proportions = proportions / proportions.sum()
        proportions = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]

        splits = np.split(idx_c, proportions)
        for i in range(num_clients):
            client_data[i]['x'].extend(x_pca[splits[i]])
            client_data[i]['y'].extend(y_train[splits[i]])

    for i in range(num_clients):
        # FIX 1: Handle clients that received 0 data due to extreme non-IID skew
        if len(client_data[i]['x']) == 0:
            client_data[i]['x'] = torch.empty((0, n_components), dtype=torch.float32)
            client_data[i]['y'] = torch.empty((0,), dtype=torch.long)
            continue

        # FIX 2: Convert list of arrays to a single numpy array first to clear the performance warning
        np_x = np.array(client_data[i]['x'])
        client_data[i]['x'] = torch.tensor(np_x, dtype=torch.float32)

        # Scale for amplitude encoding (L2 norm = 1)
        norms = torch.linalg.norm(client_data[i]['x'], dim=1, keepdim=True)
        client_data[i]['x'] = client_data[i]['x'] / (norms + 1e-8)

        client_data[i]['y'] = torch.tensor(client_data[i]['y'], dtype=torch.long)

    print(f"Data successfully partitioned into {num_clients} disjoint nodes.")
    return client_data

client_datasets = prepare_non_iid_cifar10(NUM_CLIENTS, DIRICHLET_ALPHA, N_FEATURES)

Applying PCA to reduce classical features to 256 dimensions...
Data successfully partitioned into 10 disjoint nodes.


In [5]:
# Utilizing density matrix simulator for hardware noise modeling
dev = qml.device("default.mixed", wires=N_QUBITS)

def conv_block(params, wires):
    """Translationally invariant parameterized 2-qubit unitary."""
    qml.RY(params[0], wires=wires[0])
    qml.RY(params[1], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])

    # Hardware Decoherence Simulation (p=0.01 per CNOT)
    qml.DepolarizingChannel(0.01, wires=wires[0])
    qml.DepolarizingChannel(0.01, wires=wires[1])

def pooling_block(params, wires):
    """Spatial reduction via controlled rotations and partial tracing."""
    qml.CRZ(params[0], wires=[wires[0], wires[1]])
    qml.PauliX(wires=wires[0])
    qml.CRX(params[1], wires=[wires[0], wires[1]])

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def qcnn_circuit(inputs, weights):
    # 1. Deterministic Amplitude Encoding
    qml.AmplitudeEmbedding(features=inputs, wires=range(N_QUBITS), normalize=True)

    # 2. Convolutional Layer
    weight_idx = 0
    for i in range(0, N_QUBITS - 1, 2):
        conv_block(weights[weight_idx:weight_idx+2], wires=[i, i+1])
    weight_idx += 2

    # 3. Pooling Layer (Tracing out half the qubits geometrically)
    for i in range(0, N_QUBITS - 1, 2):
        pooling_block(weights[weight_idx:weight_idx+2], wires=[i, i+1])
    weight_idx += 2

    # Measure expectation value of the surviving central qubit
    return qml.expval(qml.PauliZ(N_QUBITS - 1))

class HA_QFed_Model(nn.Module):
    def __init__(self):
        super(HA_QFed_Model, self).__init__()
        # 4 parameters per layer logic (simplified for prototype)
        self.q_params = nn.Parameter(torch.randn(4) * 0.1)

    def forward(self, x):
        batch_size = x.shape[0]
        outputs = []
        for i in range(batch_size):
            # Scale output to [0, 1] for CrossEntropy/BCE
            q_out = qcnn_circuit(x[i], self.q_params)
            outputs.append((q_out + 1.0) / 2.0)
        return torch.stack(outputs).unsqueeze(1)

In [11]:
def local_proximal_update(model, global_weights, data_x, data_y, mu, epochs, lr, batch_size=16):
    """Algorithm 1: Proximal Local Update (Edge Node execution) with Mini-Batching"""
    model.load_state_dict(copy.deepcopy(global_weights))
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    model.train()
    for epoch in range(epochs):
        # --- FIX: Mini-batching to prevent quantum simulator from freezing ---
        # Randomly sample a small batch of data for this epoch
        indices = torch.randperm(len(data_x))[:batch_size]
        batch_x = data_x[indices]
        batch_y = data_y[indices]

        optimizer.zero_grad()

        # Ensure quantum outputs are strictly float32 and clamped to [0, 1]
        outputs = model(batch_x).squeeze().to(torch.float32).clamp(0.0, 1.0)
        target_y = batch_y.to(torch.float32)

        # 1. Standard Empirical Loss (Evaluated via Parameter-Shift on QPU)
        empirical_loss = criterion(outputs, target_y)

        # 2. Classical Proximal Penalty Computation
        proximal_term = 0.0
        for param, global_param in zip(model.parameters(), global_weights.values()):
            proximal_term += ((param - global_param) ** 2).sum()

        # 3. Constrained Update
        total_loss = empirical_loss + (mu / 2.0) * proximal_term
        total_loss.backward()
        optimizer.step()

    return model.state_dict()

def federated_aggregation(local_weights_list):
    """Eq 19: Euclidean Aggregation of proximal-constrained vectors."""
    global_weights = copy.deepcopy(local_weights_list[0])
    for key in global_weights.keys():
        for i in range(1, len(local_weights_list)):
            global_weights[key] += local_weights_list[i][key]
        global_weights[key] = torch.div(global_weights[key], len(local_weights_list))
    return global_weights

In [12]:
# Change this:
# dev = qml.device("default.mixed", wires=N_QUBITS)

# To this for rapid testing:
dev = qml.device("default.qubit", wires=N_QUBITS)

In [ ]:
# Initialize Global Model
global_model = HA_QFed_Model()
global_weights = global_model.state_dict()

# Storage for convergence metrics
ha_qfed_loss_history = []

print("Initiating HA-QFed Training across Non-IID Edge Networks...")

for round in range(GLOBAL_ROUNDS):
    local_weights = []
    round_loss = 0.0

    # Dynamic Proximal Tuning (Annealing up to MAX_MU as described in Section V.F)
    current_mu = min(PROXIMAL_MU, PROXIMAL_MU * (round / 20.0)) if round < 20 else PROXIMAL_MU

    for client_id in range(NUM_CLIENTS):
        # Retrieve non-IID local data partition
        x_local = client_datasets[client_id]['x']
        y_local = client_datasets[client_id]['y']

        if len(x_local) == 0: continue # Skip if edge node is completely empty

        # Execute Algorithm 1 on the edge node
        updated_weights = local_proximal_update(
            model=HA_QFed_Model(),
            global_weights=global_weights,
            data_x=x_local,
            data_y=y_local,
            mu=current_mu,
            epochs=LOCAL_EPOCHS,
            lr=LEARNING_RATE
        )
        local_weights.append(updated_weights)

    # Server Aggregation
    global_weights = federated_aggregation(local_weights)
    global_model.load_state_dict(global_weights)

    # Escaped the backslash (\\mu) to avoid Python 3.12 SyntaxWarnings
    print(f"Global Synchronization Round {round+1}/{GLOBAL_ROUNDS} Complete | Dynamic Proximal \\mu: {current_mu:.3f}")

print("HA-QFed Optimization Complete.")

Initiating HA-QFed Training across Non-IID Edge Networks...
Global Synchronization Round 1/50 Complete | Dynamic Proximal \mu: 0.000
Global Synchronization Round 2/50 Complete | Dynamic Proximal \mu: 0.007
Global Synchronization Round 3/50 Complete | Dynamic Proximal \mu: 0.015
Global Synchronization Round 4/50 Complete | Dynamic Proximal \mu: 0.022
Global Synchronization Round 5/50 Complete | Dynamic Proximal \mu: 0.030
Global Synchronization Round 6/50 Complete | Dynamic Proximal \mu: 0.037
Global Synchronization Round 7/50 Complete | Dynamic Proximal \mu: 0.045
Global Synchronization Round 8/50 Complete | Dynamic Proximal \mu: 0.052
Global Synchronization Round 9/50 Complete | Dynamic Proximal \mu: 0.060
Global Synchronization Round 10/50 Complete | Dynamic Proximal \mu: 0.068
Global Synchronization Round 11/50 Complete | Dynamic Proximal \mu: 0.075
Global Synchronization Round 12/50 Complete | Dynamic Proximal \mu: 0.083
Global Synchronization Round 13/50 Complete | Dynamic Proxima